# Experiment: Macro State Sigma

Objective:
- Reload every retained stage-2 macro model under `loc_model_stage2/stage2_macro`.
- Compute the macro `sigmas_matrix` for each saved scale from the checkpoint itself.
- Derive `log(det(sigmas_matrix))` and `term1` from each regenerated sigma matrix using the same formula as `EI_calculation.py`.
- Save the regenerated matrices and `term1` values for later comparison or visualization.


In [1]:
from __future__ import annotations

import math
import re
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import torch
from torch import nn

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "models_macro.py").exists() and (candidate / "src" / "models_micro.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.data_sources import load_array_from_path
from src.models_macro import Parellel_Renorm_Dynamic as MacroParellelRenormDynamic
from src.models_macro import _build_windows_from_series as build_macro_windows_from_series
from src.models_micro import Parellel_Renorm_Dynamic as MicroParellelRenormDynamic
from src.models_micro import _build_windows_from_series as build_micro_windows_from_series

torch.set_grad_enabled(False)
DEFAULT_RUN_NAME = "stage2_macro"
DEFAULT_DATA_PATH = Path("loc_data_kuramoto") / "generated_data.npz"


class CompatibleMacroParellelRenormDynamic(MacroParellelRenormDynamic):
    """Allow reconstructing legacy checkpoints that keep a same-dimension macro scale."""

    def _build_scale_dims(self, reduce_dims, group):
        reduce_dim_schedule = self._resolve_reduce_dims(reduce_dims)
        if len(reduce_dim_schedule) == 1 and int(reduce_dim_schedule[0]) == self.sym_size:
            return (
                [self.sym_size],
                [{"type": "dense", "input_dim": self.sym_size, "output_dim": self.sym_size}],
                reduce_dim_schedule,
            )
        return super()._build_scale_dims(reduce_dims, group)


In [2]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "models_macro.py").exists() and (candidate / "src" / "models_micro.py").exists() and (candidate / "loc_model_stage2").exists():
            return candidate
    raise FileNotFoundError("Could not locate the causal_network_mix_2_0.2_syn2 project root.")


def parse_reduce_dims(value) -> List[int]:
    if value is None:
        return []
    if isinstance(value, float) and math.isnan(value):
        return []
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return []
    return [int(part.strip()) for part in text.split(",") if part.strip()]


def parse_scale_dims(value) -> List[int]:
    return parse_reduce_dims(value)


def extract_index(path: Path, pattern: str) -> Optional[int]:
    match = re.search(pattern, path.name)
    return None if match is None else int(match.group(1))


def list_stage2_scale_artifacts(project_root: Path, run_name: str = DEFAULT_RUN_NAME) -> List[Dict]:
    model_dir = project_root / "loc_model_stage2" / run_name
    result_dir = project_root / "loc_result_stage2" / run_name
    if not model_dir.exists():
        raise FileNotFoundError(f"Missing model directory: {model_dir}")
    if not result_dir.exists():
        raise FileNotFoundError(f"Missing result directory: {result_dir}")

    model_map = {
        idx: path
        for path in sorted(model_dir.glob("model_scale*.pkl"))
        for idx in [extract_index(path, r"model_scale(\d+)\.pkl")]
        if idx is not None
    }
    summary_map = {
        idx: path
        for path in sorted(result_dir.glob("summary_scale*.csv"))
        for idx in [extract_index(path, r"summary_scale(\d+)\.csv")]
        if idx is not None
    }

    artifacts = []
    for file_scale in sorted(set(model_map) | set(summary_map)):
        model_path = model_map.get(file_scale)
        summary_path = summary_map.get(file_scale)
        if model_path is None or summary_path is None:
            continue

        summary_row = pd.read_csv(summary_path).iloc[0].to_dict()
        logical_scale_id = int(summary_row.get("scale_id", file_scale))
        scale_dims = parse_scale_dims(summary_row.get("scale_dims", ""))
        reduce_dims = parse_reduce_dims(summary_row.get("reduce_dims", ""))
        scale_dim = None
        if scale_dims and 0 <= logical_scale_id < len(scale_dims):
            scale_dim = int(scale_dims[logical_scale_id])
        elif reduce_dims:
            if file_scale == 0 and len(reduce_dims) == 1:
                scale_dim = int(reduce_dims[0])
            elif 0 <= logical_scale_id < len(reduce_dims):
                scale_dim = int(reduce_dims[logical_scale_id])

        artifacts.append(
            {
                "file_scale": int(file_scale),
                "model_family": "micro" if int(file_scale) == 0 else "macro",
                "logical_scale_id": logical_scale_id,
                "scale_dim": scale_dim,
                "scale_dims": scale_dims,
                "reduce_dims": reduce_dims,
                "group": parse_reduce_dims(summary_row.get("group", "")),
                "hidden_units1": int(summary_row.get("hidden_units1", 100)),
                "hidden_units2": int(summary_row.get("hidden_units2", 100)),
                "flow_num_layers": int(summary_row.get("flow_num_layers", 3)),
                "dynamics_num_layers": int(summary_row.get("dynamics_num_layers", 4)),
                "latent_size": int(summary_row.get("latent_size", 1)),
                "encoder_type": "mlp" if str(summary_row.get("encoder_type", "mlp")).strip().lower() in {"", "nan"} else str(summary_row.get("encoder_type", "mlp")).strip(),
                "time_delay": int(summary_row.get("time_delay", 1)),
                "batch_size": int(summary_row.get("batch_size", 1024)),
                "model_path": model_path,
                "summary_path": summary_path,
            }
        )
    if not artifacts:
        raise FileNotFoundError("No paired stage2 macro summary/model artifacts were found.")
    return artifacts


def load_stage2_origin_data(project_root: Path, data_path: Optional[Path] = None) -> np.ndarray:
    resolved_path = project_root / (data_path or DEFAULT_DATA_PATH)
    origin_data = load_array_from_path(str(resolved_path))
    if origin_data is None:
        raise FileNotFoundError(f"Could not load origin data from: {resolved_path}")
    return np.asarray(origin_data, dtype=np.float32)


def build_sigma_windows(origin_data: np.ndarray, time_delay: int, model_family: str, max_samples: Optional[int] = None, use_train_split: bool = True) -> Tuple[np.ndarray, np.ndarray]:
    window_builder = build_micro_windows_from_series if model_family == "micro" else build_macro_windows_from_series
    x_all, y_all = window_builder(origin_data, int(time_delay))
    if use_train_split:
        train_end = max(1, int(len(x_all) * 0.95))
        x_all = x_all[:train_end]
        y_all = y_all[:train_end]
    if max_samples is not None:
        max_samples = int(max_samples)
        x_all = x_all[:max_samples]
        y_all = y_all[:max_samples]
    if len(x_all) == 0:
        raise ValueError("No windows were generated for sigma estimation.")
    return x_all.astype(np.float32), y_all.astype(np.float32)


def build_macro_model(num_nodes: int, artifact: dict, device: torch.device):
    model_cls = MicroParellelRenormDynamic if artifact["model_family"] == "micro" else CompatibleMacroParellelRenormDynamic
    model = model_cls(
        sym_size=int(num_nodes),
        latent_size=int(artifact["latent_size"]),
        effect_size=int(num_nodes),
        cut_size=2,
        hidden_units1=int(artifact["hidden_units1"]),
        hidden_units2=int(artifact["hidden_units2"]),
        normalized_state=True,
        device=device,
        is_random=False,
        flow_num_layers=int(artifact["flow_num_layers"]),
        dynamics_num_layers=int(artifact["dynamics_num_layers"]),
        decode_noise_scale=0.0,
        reduce_dims=artifact["reduce_dims"],
        group=artifact["group"],
        encoder_type=artifact["encoder_type"],
    ).to(device)
    state_dict = torch.load(artifact["model_path"], map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def compute_log_det_from_sigmas_matrix(sigmas_matrix: Union[np.ndarray, torch.Tensor]) -> float:
    sigma_tensor = torch.as_tensor(sigmas_matrix, dtype=torch.float32)
    if sigma_tensor.ndim != 2 or sigma_tensor.shape[0] != sigma_tensor.shape[1]:
        raise ValueError("sigmas_matrix must be a square matrix.")
    if int(sigma_tensor.shape[0]) == 0:
        raise ValueError("sigmas_matrix must be non-empty.")
    sign, logabsdet = torch.linalg.slogdet(sigma_tensor)
    if not torch.isfinite(logabsdet):
        raise ValueError("log(det(sigmas_matrix)) is not finite.")
    if float(sign.item()) <= 0.0:
        raise ValueError("sigmas_matrix must have a positive determinant to compute log(det(sigmas_matrix)).")
    return float(logabsdet.item())


def compute_term1_from_log_det(log_det_sigmas_matrix: float, output_size: int) -> float:
    output_size = int(output_size)
    if output_size < 1:
        raise ValueError("output_size must be positive.")
    term1 = - (1 + np.log(2 * np.pi) + float(log_det_sigmas_matrix) / output_size) / 2
    return float(term1)


def compute_macro_sigma_for_scale(
    artifact: dict,
    origin_data: np.ndarray,
    device: Union[str, torch.device] = "cpu",
    max_samples: Optional[int] = None,
    use_train_split: bool = True,
):
    device = torch.device(device)
    model = build_macro_model(origin_data.shape[-1], artifact, device=device)
    x_np, y_np = build_sigma_windows(
        origin_data,
        time_delay=artifact["time_delay"],
        model_family=artifact["model_family"],
        max_samples=max_samples,
        use_train_split=use_train_split,
    )
    x_tensor = torch.tensor(x_np, dtype=torch.float32, device=device)
    y_tensor = torch.tensor(y_np, dtype=torch.float32, device=device)
    mse_raw = nn.MSELoss(reduction="none")
    sigmas, sigmas_matrix, _, _ = model.estimate_sigmas_matrix(
        x_tensor,
        y_tensor,
        artifact["logical_scale_id"],
        mse_raw,
    )
    log_det_sigmas_matrix = compute_log_det_from_sigmas_matrix(sigmas_matrix)
    term1 = compute_term1_from_log_det(log_det_sigmas_matrix, sigmas_matrix.shape[0])
    return {
        **artifact,
        "num_windows": int(len(x_np)),
        "log_det_sigmas_matrix": log_det_sigmas_matrix,
        "term1": term1,
        "sigmas": sigmas.detach().cpu().numpy(),
        "sigmas_matrix": sigmas_matrix.detach().cpu().numpy(),
    }


def save_macro_sigma_results(
    results: List[Dict],
    output_dir: Path,
    file_name_template: str = "macro_sigmas_matrix_scale{file_scale}.csv",
) -> List[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_paths = []
    for item in results:
        output_path = output_dir / file_name_template.format(file_scale=item["file_scale"])
        pd.DataFrame(item["sigmas_matrix"]).to_csv(output_path, index=False)
        saved_paths.append(output_path)
    return saved_paths


def save_macro_term1_results(
    results: List[Dict],
    output_dir: Path,
    file_name_template: str = "term1_scale{file_scale}.csv",
) -> List[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_paths = []
    for item in results:
        output_path = output_dir / file_name_template.format(file_scale=item["file_scale"])
        pd.DataFrame([[item["term1"]]]).to_csv(output_path, index=False, header=False)
        saved_paths.append(output_path)
    return saved_paths


def compute_macro_sigmas_for_all_scales(
    project_root: Optional[Path] = None,
    run_name: str = DEFAULT_RUN_NAME,
    data_path: Optional[Path] = None,
    device: Union[str, torch.device] = "cpu",
    max_samples: Optional[int] = None,
    use_train_split: bool = True,
    output_dir: Optional[Path] = None,
):
    project_root = find_project_root(project_root)
    origin_data = load_stage2_origin_data(project_root, data_path=data_path)
    artifacts = list_stage2_scale_artifacts(project_root, run_name=run_name)
    results = [
        compute_macro_sigma_for_scale(
            artifact,
            origin_data,
            device=device,
            max_samples=max_samples,
            use_train_split=use_train_split,
        )
        for artifact in artifacts
    ]

    if output_dir is None:
        output_dir = project_root / "loc_result_stage2" / run_name / "macro_state_sig"
    output_dir = Path(output_dir)
    sigma_paths = save_macro_sigma_results(results, output_dir=output_dir)
    term1_paths = save_macro_term1_results(results, output_dir=output_dir)

    summary_rows = []
    for item, sigma_path, term1_path in zip(results, sigma_paths, term1_paths):
        summary_rows.append(
            {
                "file_scale": item["file_scale"],
                "model_family": item["model_family"],
                "logical_scale_id": item["logical_scale_id"],
                "scale_dim": item["scale_dim"],
                "time_delay": item["time_delay"],
                "num_windows": item["num_windows"],
                "log_det_sigmas_matrix": item["log_det_sigmas_matrix"],
                "term1": item["term1"],
                "model_path": str(item["model_path"].resolve()),
                "summary_path": str(item["summary_path"].resolve()),
                "sigmas_csv": str(sigma_path.resolve()),
                "term1_csv": str(term1_path.resolve()),
            }
        )
    return pd.DataFrame(summary_rows).sort_values("file_scale").reset_index(drop=True), results


In [3]:
PROJECT_ROOT = find_project_root()
SIGMA_SUMMARY_DF, SIGMA_RESULTS = compute_macro_sigmas_for_all_scales(
    project_root=PROJECT_ROOT,
    run_name=DEFAULT_RUN_NAME,
    data_path=DEFAULT_DATA_PATH,
    device="cpu",
    max_samples=2048,
    use_train_split=True,
)
SIGMA_SUMMARY_DF

,file_scale,model_family,logical_scale_id,scale_dim,time_delay,num_windows,log_det_sigmas_matrix,term1,model_path,summary_path,sigmas_csv,term1_csv
0,0,micro,0,32,1,2048,-297.438385,3.228536,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
1,1,macro,0,32,1,2048,-288.239624,3.084806,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
2,2,macro,1,16,1,2048,-108.324783,1.966211,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
3,3,macro,2,8,1,2048,-53.752995,1.940624,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
4,4,macro,3,4,1,2048,-31.033531,2.460253,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
5,5,macro,4,2,1,2048,-16.433916,2.689540,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...
6,6,macro,5,1,1,2048,-3.934427,0.548275,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...,/home/wangzhipeng/code/causal_network/causal_n...


In [4]:
sigma_diag_summary = pd.DataFrame(
    [
        {
            "file_scale": item["file_scale"],
            "model_family": item["model_family"],
            "scale_dim": item["scale_dim"],
            "log_det_sigmas_matrix": item["log_det_sigmas_matrix"],
            "term1": item["term1"],
            **{f"sigma_{idx}": value for idx, value in enumerate(item["sigmas"])},
        }
        for item in SIGMA_RESULTS
    ]
).sort_values("file_scale").reset_index(drop=True)
sigma_diag_summary


,file_scale,model_family,scale_dim,log_det_sigmas_matrix,term1,sigma_0,sigma_1,sigma_2,sigma_3,sigma_4,...,sigma_22,sigma_23,sigma_24,sigma_25,sigma_26,sigma_27,sigma_28,sigma_29,sigma_30,sigma_31
0,0,micro,32,-297.438385,3.228536,0.000057,0.000045,0.000052,0.000655,0.000085,...,0.000070,0.000031,0.000089,0.000051,0.000472,0.000186,0.000111,0.000116,0.000042,0.000036
1,1,macro,32,-288.239624,3.084806,0.000185,0.000276,0.000106,0.000025,0.000060,...,0.000132,0.000137,0.000075,0.000215,0.000175,0.000176,0.000108,0.000130,0.000101,0.000113
2,2,macro,16,-108.324783,1.966211,0.000723,0.001405,0.003981,0.000395,0.001274,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,macro,8,-53.752995,1.940624,0.001199,0.001049,0.001631,0.000235,0.000944,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,macro,4,-31.033531,2.460253,0.000241,0.000304,0.001482,0.000307,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,5,macro,2,-16.433916,2.689540,0.000266,0.000274,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,6,macro,1,-3.934427,0.548275,0.019557,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
term1_summary = pd.DataFrame(
    [
        {
            "file_scale": item["file_scale"],
            "logical_scale_id": item["logical_scale_id"],
            "scale_dim": item["scale_dim"],
            "log_det_sigmas_matrix": item["log_det_sigmas_matrix"],
            "term1": item["term1"],
        }
        for item in SIGMA_RESULTS
    ]
).sort_values("file_scale").reset_index(drop=True)
term1_summary


,file_scale,logical_scale_id,scale_dim,log_det_sigmas_matrix,term1
0,0,0,32,-297.438385,3.228536
1,1,0,32,-288.239624,3.084806
2,2,1,16,-108.324783,1.966211
3,3,2,8,-53.752995,1.940624
4,4,3,4,-31.033531,2.460253
5,5,4,2,-16.433916,2.689540
6,6,5,1,-3.934427,0.548275
